In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
# exp_name = 'friction-walking-fractal-kp2000kd50'
# exp_name = 'friction-walking-flat-kp2000kd50'  # ckpt = 20000
# exp_name = 'friction-walking-fractal-kp2000kd50-action_rate-0.01'  # ckpt = 20000
exp_name = 'correct-walking-flat-kp2000kd50'
# exp_name = 'friction-walking-fractal-kp2000kd50-linvel0.8-pos3'
ckpt = 300

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.9
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.9,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.1, 2.0],
  'restitution': [0.0, 0.2],
  'kp': [15000.0, 25000.0],
  'kd': [40.0, 80.0]

In [10]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [11]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [12]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[ 0.1121,  0.2666, -0.0162,  0.3496, -0.4614,  0.1429, -0.1820, -0.0485,
         -0.1202, -0.3984, -0.6206, -0.1023]], device='cuda:0')
Scaled actions :  tensor([[ 0.1121,  0.2666, -0.0162,  0.3496, -0.4614,  0.1429, -0.1820, -0.0485,
         -0.1202, -0.3984, -0.6206, -0.1023]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[ 1.2169e-05, -1.2488e-02,  1.7362e-05,  1.1243e-10, -7.4466e-21,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.1109e-07,
         -1.5817e-07, -4.7266e-05,  2.2531e-04, -1.2803e-04,  2.3233e-07,
          1.6094e-07, -7.5484e-09, -4.7386e-05,  2.2554e-04, -1.2821e-04,
         -1.3234e-06, -5.5543e-06, -7.9086e-06, -2.3640e-03,  1.1269e-02,
         -6.4021e-03,  1.1617e-05,  8.0469e-06, -3.7742e-07, -2.3695e-03,
          1.1281e-02, -6.4100e-03, -6.6171e-05,  1.1208e-01,  2.6657e-01,
         -1.6237e-02,  3.4958e-01, -4.6144e-01,  1.4286e-01, -1.8201e-01,
         -4.8523e-02, -1.2023e-01, -3.9837e-01, -6.2064e-01, -1.0230e-01]],
       device='cuda:0')
torques: [-8.00762319e-17  3.59403292e-16 -1.30992703e-05  2.93225639e-05
 -1.25000976e-05  2.71341572e-16  7.79769018e-17 -1.74006023e-16
 -1.30992703e-05  2.93225639e-05 -1.25000976e-05 -3.22752275e-17]
データ収集: step 2


In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[ 0.0810,  0.1336,  0.4836, -0.9803, -0.1627, -0.1311,  0.1861, -0.9278,
          0.7128,  0.1589,  0.2007,  0.2951]], device='cuda:0')
Scaled actions :  tensor([[ 0.0810,  0.1336,  0.4836, -0.9803, -0.1627, -0.1311,  0.1861, -0.9278,
          0.7128,  0.1589,  0.2007,  0.2951]], device='cuda:0')
obs :  tensor([[-0.0697,  0.1099, -0.0671,  0.0023,  0.0016, -1.0000,  1.0000,  0.0000,
          0.0000,  0.0099,  0.0055, -0.0054,  0.0097, -0.0137,  0.0118, -0.0102,
          0.0020, -0.0041,  0.0050, -0.0111, -0.0089,  0.0812,  0.0496, -0.0430,
          0.0761, -0.1181,  0.1022, -0.0923,  0.0169, -0.0325,  0.0337, -0.0942,
         -0.0733,  0.0810,  0.1336,  0.4836, -0.9803, -0.1627, -0.1311,  0.1861,
         -0.9278,  0.7128,  0.1589,  0.2007,  0.2951]], device='cuda:0')
torques: [  57.14962077  200.           55.16985976  200.         -200.
   81.52483825 -189.30735895 -131.43846977 -174.2835154  -200.
 -200.          -53.88800563]
データ収集: step 3


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[-0.5212, -0.9829, -0.7578, -0.0263,  0.0072, -0.0097,  0.2156,  0.8483,
         -1.0292, -0.4335, -0.2272,  0.0745]], device='cuda:0')
Scaled actions :  tensor([[-0.5212, -0.9829, -0.7578, -0.0263,  0.0072, -0.0097,  0.2156,  0.8483,
         -1.0292, -0.4335, -0.2272,  0.0745]], device='cuda:0')
obs :  tensor([[-6.2729e-02, -1.3409e-01, -6.4637e-02,  1.6984e-03,  4.0806e-03,
         -9.9999e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.9172e-02,
          1.8391e-02, -2.4692e-03,  1.7542e-02, -4.4970e-02,  1.9337e-02,
         -1.7106e-02, -2.2666e-04, -3.4887e-03,  1.8562e-02, -1.7842e-02,
         -1.1188e-02,  1.0732e-01,  5.9293e-02,  5.5987e-02,  3.6395e-02,
         -1.7456e-01, -2.6613e-02,  1.2460e-02, -3.3411e-02,  3.7349e-02,
          9.4617e-02,  1.5373e-02,  3.9207e-02, -5.2119e-01, -9.8288e-01,
         -7.5779e-01, -2.6328e-02,  7.2039e-03, -9.7102e-03,  2.1564e-01,
          8.4835e-01, -1.0292e+00, -4.3354e-01, -2.2721e-01,  7.4

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.0965, -0.2063,  1.1485, -0.1062,  0.1596,  0.2279,  0.0343,  0.2334,
          1.1763, -0.2135,  0.1877, -0.1759]], device='cuda:0')
Scaled actions :  tensor([[-0.0965, -0.2063,  1.1485, -0.1062,  0.1596,  0.2279,  0.0343,  0.2334,
          1.1763, -0.2135,  0.1877, -0.1759]], device='cuda:0')
obs :  tensor([[-0.0505,  0.1149, -0.0907,  0.0018,  0.0066, -1.0000,  1.0000,  0.0000,
          0.0000,  0.0409,  0.0251, -0.0015,  0.0236, -0.0673,  0.0118, -0.0042,
         -0.0029, -0.0033,  0.0298, -0.0268,  0.0030,  0.0155,  0.0185, -0.0364,
          0.0261, -0.0598, -0.0421,  0.1070,  0.0016, -0.0293,  0.0252, -0.0941,
          0.0903, -0.0965, -0.2063,  1.1485, -0.1062,  0.1596,  0.2279,  0.0343,
          0.2334,  1.1763, -0.2135,  0.1877, -0.1759]], device='cuda:0')
torques: [-200.         -200.         -200.         -125.77172221  200.
   -2.88916007  200.          200.         -200.         -200.
 -200.           59.47911026]
データ収集: step 5


In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-0.1628, -0.1442, -0.8148,  0.2011,  0.1426,  0.4411, -0.2738,  0.1267,
         -0.9156,  0.3047,  0.2021, -0.3560]], device='cuda:0')
Scaled actions :  tensor([[-0.1628, -0.1442, -0.8148,  0.2011,  0.1426,  0.4411, -0.2738,  0.1267,
         -0.9156,  0.3047,  0.2021, -0.3560]], device='cuda:0')
obs :  tensor([[ 2.0110e-02, -1.0335e-01, -9.3254e-02,  1.5689e-03,  7.0509e-03,
         -9.9997e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  3.3638e-02,
          2.4499e-02, -1.9315e-03,  2.5908e-02, -6.6835e-02,  1.4117e-02,
          1.5074e-02,  8.6492e-04,  2.9101e-03,  2.5040e-02, -3.3434e-02,
          8.7124e-03, -7.5801e-02, -2.0992e-02,  2.6058e-02,  1.0868e-07,
          5.2697e-02,  5.5135e-02,  8.5260e-02,  3.2881e-02,  8.0194e-02,
         -6.4212e-02,  1.6776e-02, -2.2173e-02, -1.6282e-01, -1.4422e-01,
         -8.1481e-01,  2.0115e-01,  1.4263e-01,  4.4110e-01, -2.7381e-01,
          1.2675e-01, -9.1557e-01,  3.0469e-01,  2.0208e-01, -3.5

In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[ 0.5325,  1.2529,  1.4495, -1.2936, -0.1457,  0.0181, -0.3306, -1.2163,
          0.7933, -0.5781, -0.0269,  0.0968]], device='cuda:0')
Scaled actions :  tensor([[ 0.5325,  1.2529,  1.4495, -1.2936, -0.1457,  0.0181, -0.3306, -1.2163,
          0.7933, -0.5781, -0.0269,  0.0968]], device='cuda:0')
obs :  tensor([[ 4.0970e-02,  3.0266e-02,  2.7425e-02,  3.8315e-04,  5.8455e-03,
         -9.9998e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  8.3300e-03,
          1.5342e-02, -7.4289e-03,  3.5545e-02, -4.4107e-02,  3.7445e-02,
          2.0362e-02,  1.1890e-02,  8.7941e-03,  2.2095e-02, -1.7889e-02,
         -8.0971e-03, -1.6542e-01, -6.6499e-02, -7.1471e-02,  8.7385e-02,
          1.6350e-01,  1.6696e-01, -2.1506e-02,  7.1584e-02, -1.1828e-02,
          2.5905e-02,  1.2759e-01, -1.3465e-01,  5.3246e-01,  1.2529e+00,
          1.4495e+00, -1.2936e+00, -1.4568e-01,  1.8062e-02, -3.3057e-01,
         -1.2163e+00,  7.9325e-01, -5.7811e-01, -2.6903e-02,  9.6

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 0.4102,  0.6295, -0.6401, -0.2227, -0.0573, -0.0326,  0.0844, -0.8835,
         -1.2824,  0.6902, -0.3237, -0.0407]], device='cuda:0')
Scaled actions :  tensor([[ 0.4102,  0.6295, -0.6401, -0.2227, -0.0573, -0.0326,  0.0844, -0.8835,
         -1.2824,  0.6902, -0.3237, -0.0407]], device='cuda:0')
obs :  tensor([[-0.0574, -0.1566, -0.0120, -0.0028,  0.0074, -1.0000,  1.0000,  0.0000,
          0.0000, -0.0140,  0.0093, -0.0098,  0.0426, -0.0236,  0.0610,  0.0043,
          0.0196,  0.0115,  0.0278, -0.0087, -0.0203, -0.0690, -0.0097,  0.0337,
         -0.0063,  0.0529,  0.0797, -0.1230,  0.0267,  0.0440,  0.0118,  0.0061,
         -0.0035,  0.4102,  0.6295, -0.6401, -0.2227, -0.0573, -0.0326,  0.0844,
         -0.8835, -1.2824,  0.6902, -0.3237, -0.0407]], device='cuda:0')
torques: [ 200.          200.          200.         -200.         -200.
 -171.96578255 -200.         -200.          200.         -200.
  -49.74723365  200.        ]
データ収集: step 8


In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 3.5173e-01,  4.1038e-01,  1.4029e+00,  1.7959e-01, -8.8338e-02,
         -3.2042e-01, -1.9933e-01, -8.3210e-01,  5.9567e-01, -1.3735e+00,
          8.5317e-04,  1.6568e-01]], device='cuda:0')
Scaled actions :  tensor([[ 3.5173e-01,  4.1038e-01,  1.4029e+00,  1.7959e-01, -8.8338e-02,
         -3.2042e-01, -1.9933e-01, -8.3210e-01,  5.9567e-01, -1.3735e+00,
          8.5317e-04,  1.6568e-01]], device='cuda:0')
obs :  tensor([[-1.6602e-02,  8.1405e-02, -1.0511e-01, -4.0192e-03,  8.9736e-03,
         -9.9995e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.7660e-02,
          1.0986e-02, -9.5269e-03,  3.9161e-02, -2.0815e-02,  6.3696e-02,
         -9.7390e-03,  1.9374e-02,  8.2347e-03,  3.8677e-02, -1.9662e-02,
         -2.2831e-02,  2.0734e-02,  1.8888e-02, -2.2789e-02, -6.8413e-03,
         -2.6367e-02, -4.6032e-02, -2.6221e-02, -2.5125e-02, -6.6876e-02,
          8.7055e-02, -1.0487e-01, -2.0839e-02,  3.5173e-01,  4.1038e-01,
          1.4029e+00,  1.

In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-0.1360, -0.8781, -0.3604, -0.3878, -0.1122,  0.1378,  0.2451,  0.8873,
         -1.4446, -0.5342,  0.0536,  0.0341]], device='cuda:0')
Scaled actions :  tensor([[-0.1360, -0.8781, -0.3604, -0.3878, -0.1122,  0.1378,  0.2451,  0.8873,
         -1.4446, -0.5342,  0.0536,  0.0341]], device='cuda:0')
obs :  tensor([[-0.0437, -0.1323, -0.1505, -0.0055,  0.0102, -0.9999,  1.0000,  0.0000,
          0.0000, -0.0031,  0.0196, -0.0070,  0.0447, -0.0324,  0.0422, -0.0253,
          0.0104,  0.0064,  0.0470, -0.0339, -0.0147,  0.1150,  0.0627,  0.0421,
          0.0561, -0.0738, -0.1579, -0.1196, -0.0612,  0.0379,  0.0047, -0.0396,
          0.0910, -0.1360, -0.8781, -0.3604, -0.3878, -0.1122,  0.1378,  0.2451,
          0.8873, -1.4446, -0.5342,  0.0536,  0.0341]], device='cuda:0')
torques: [ 200.          200.          200.          200.          -43.57143468
 -200.         -200.         -200.          200.         -200.
  113.74596773  200.        ]
データ収集:

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[-0.3888, -0.3952,  1.2569,  0.7798,  0.0598,  0.1522,  0.2276,  0.4860,
          1.0554,  0.1984,  0.1027, -0.3733]], device='cuda:0')
Scaled actions :  tensor([[-0.3888, -0.3952,  1.2569,  0.7798,  0.0598,  0.1522,  0.2276,  0.4860,
          1.0554,  0.1984,  0.1027, -0.3733]], device='cuda:0')
obs :  tensor([[-0.0407,  0.1531, -0.1375, -0.0046,  0.0119, -0.9999,  1.0000,  0.0000,
          0.0000,  0.0096,  0.0278, -0.0067,  0.0482, -0.0509,  0.0229, -0.0387,
          0.0025,  0.0054,  0.0401, -0.0307,  0.0032,  0.0216,  0.0228, -0.0327,
         -0.0143, -0.1026, -0.0460, -0.0245, -0.0212, -0.0399, -0.0668,  0.0538,
          0.0853, -0.3888, -0.3952,  1.2569,  0.7798,  0.0598,  0.1522,  0.2276,
          0.4860,  1.0554,  0.1984,  0.1027, -0.3733]], device='cuda:0')
torques: [-200.         -200.         -200.         -200.          -25.38437988
  200.          200.          200.         -200.         -200.
  123.99572944  -21.18505688]
データ収集

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[-0.1190, -0.2833, -0.6984, -1.0986, -0.0628,  0.3297,  0.0160,  0.2816,
         -0.9489, -0.8402, -0.0978, -0.3166]], device='cuda:0')
Scaled actions :  tensor([[-0.1190, -0.2833, -0.6984, -1.0986, -0.0628,  0.3297,  0.0160,  0.2816,
         -0.9489, -0.8402, -0.0978, -0.3166]], device='cuda:0')
obs :  tensor([[-8.1877e-02, -1.7299e-01, -1.3936e-01, -5.4868e-03,  1.4158e-02,
         -9.9988e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  3.4948e-03,
          2.8500e-02, -4.3306e-03,  5.2758e-02, -5.9374e-02,  2.5906e-02,
         -3.2979e-02,  3.7259e-03,  2.6070e-03,  4.1528e-02, -1.5833e-02,
          6.0509e-03, -7.2633e-02, -1.0511e-02,  5.0626e-02,  5.2769e-02,
          7.3108e-03,  6.2901e-02,  7.1281e-02,  2.8658e-02,  8.3309e-04,
          7.6901e-02,  7.5855e-02, -4.3989e-02, -1.1902e-01, -2.8334e-01,
         -6.9845e-01, -1.0986e+00, -6.2770e-02,  3.2967e-01,  1.5997e-02,
          2.8161e-01, -9.4894e-01, -8.4020e-01, -9.7754e-02, -3.

In [35]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=3.753, Scaled action max=3.753
Step 1/10, Total steps: 132
steps: 132
actions : tensor([[ 0.3291,  0.8131, -4.6813,  3.0911,  2.7744,  0.2538,  0.2169,  0.1199,
          3.7533, -0.9946, -2.4747, -0.6306]], device='cuda:0')
target_dof_pos: tensor([[ 0.4168,  1.0592, -5.1550,  4.9725,  2.7145,  0.3251,  0.2167,  0.1912,
          2.6015,  0.7125, -3.4804, -0.7373]], device='cuda:0')
Step 1: Original action max=4.052, Scaled action max=4.052
Step 2: Original action max=4.370, Scaled action max=4.370
データ収集完了: 10 steps collected with action_scale=1.0


In [36]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [37]:
env.sim.stop()

In [44]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.9.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-fractal-kp2000kd50_ckpt100_scale1.0_rotorInertia0.9.csv
データ形状: (192, 58)
